In [1]:
import torch
import time

https://docs.pytorch.org/docs/stable/notes/get_start_xpu.html

In [2]:
print(f"Versión de PyTorch: {torch.__version__}")

# 1. Verificación básica
disponible = torch.xpu.is_available()
print(f"¿XPU (GPU Intel) disponible?: {disponible}")

if disponible:
    # 2. Información del hardware detectado por PyTorch
    nombre_gpu = torch.xpu.get_device_name(0)
    print(f"GPU detectada: {nombre_gpu}")
    
    # 3. Prueba de fuego: Operación matemática real en la GPU
    # Creamos dos tensores grandes directamente en la XPU
    a = torch.randn(2000, 2000, device="xpu")
    b = torch.randn(2000, 2000, device="xpu")
    
    # Realizamos una multiplicación de matrices (punto de estrés)
    c = torch.matmul(a, b)
    
    print("Prueba de cálculo completada. ¡El tensor está en la GPU!")
    print(f"Ubicación del resultado: {c.device}")
else:
    print("PyTorch no ve la GPU. Verifica que instalaste con '--index-url https://download.pytorch.org/whl/xpu'")

Versión de PyTorch: 2.11.0+xpu
¿XPU (GPU Intel) disponible?: True
GPU detectada: Intel(R) Arc(TM) Graphics
Prueba de cálculo completada. ¡El tensor está en la GPU!
Ubicación del resultado: xpu:0


In [6]:
def benchmark(device_name, size=8000):
    # Definir el dispositivo
    device = torch.device(device_name)
    print(f"\n🚀 Iniciando benchmark en: {device_name.upper()}")
    
    # Crear dos matrices grandes
    # (8000x8000 es lo suficientemente grande para notar la diferencia)
    try:
        a = torch.randn(size, size, device=device)
        b = torch.randn(size, size, device=device)
        
        # Calentamiento (para que la GPU se despierte)
        _ = torch.matmul(a, b)
        if device_name == "xpu":
            torch.xpu.synchronize()
            
        # Medición real
        start_time = time.time()
        for _ in range(10):
            c = torch.matmul(a, b)
        
        if device_name == "xpu":
            torch.xpu.synchronize() # Esperar a que la GPU termine
            
        end_time = time.time()
        
        avg_time = (end_time - start_time) / 10
        print(f"✅ Tiempo promedio por operación: {avg_time:.4f} segundos")
        return avg_time
    except Exception as e:
        print(f"❌ Error en {device_name}: {e}")
        return None

# Ejecutar comparativa
if __name__ == "__main__":
    print(f"Versión de PyTorch: {torch.__version__}")
    
    t_cpu = benchmark("cpu")
    
    if torch.xpu.is_available():
        t_gpu = benchmark("xpu")
        
        if t_cpu and t_gpu:
            print(f"\n📊 RESULTADO FINAL:")
            print(f"La GPU es {t_cpu / t_gpu:.2f} veces más rápida que la CPU.")
    else:
        print("\n❌ La GPU XPU no está disponible para PyTorch.")

Versión de PyTorch: 2.11.0+xpu

🚀 Iniciando benchmark en: CPU
✅ Tiempo promedio por operación: 2.0076 segundos

🚀 Iniciando benchmark en: XPU
✅ Tiempo promedio por operación: 0.2888 segundos

📊 RESULTADO FINAL:
La GPU es 6.95 veces más rápida que la CPU.


In [2]:
import time
import torch
import pandas as pd

# -----------------------------
# Synchronization helper
# -----------------------------
def sync(device):
    """
    Sincroniza el dispositivo antes y después de medir tiempos.
    """
    device = torch.device(device)
    if device.type == "xpu":
        torch.xpu.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()
    # MPS eliminado, CPU no necesita sincronización explícita

# -----------------------------
# Core benchmark runner
# -----------------------------
def benchmark_op(op_fn, make_inputs_fn, device_str, n_warmup=10, n_iters=50):
    device = torch.device(device_str)
    inputs = make_inputs_fn(device)

    # Warmup: Es vital en Intel para que el compilador JIT haga su magia
    for _ in range(n_warmup):
        _ = op_fn(*inputs)
    sync(device)

    # Timed loop
    times = []
    out = None
    for _ in range(n_iters):
        sync(device)
        t0 = time.perf_counter()
        out = op_fn(*inputs)
        sync(device) # Sincronización post-operación
        t1 = time.perf_counter()
        times.append(t1 - t0)

    return {
        "mean_ms": 1000.0 * sum(times) / len(times),
        "min_ms": 1000.0 * min(times),
        "max_ms": 1000.0 * max(times),
        "std_ms": 1000.0 * (sum((x - sum(times)/len(times))**2 for x in times) / len(times))**0.5,
        "output_shape": tuple(out.shape) if hasattr(out, "shape") else None,
        "output": out.detach().cpu() if torch.is_tensor(out) else out,
    }

# -----------------------------
# Operation library (Se mantiene igual, ajustada para XPU)
# -----------------------------
def make_operation_specs():
    specs = []
    
    # 1. Matmul (La prueba de fuego para los Xe-cores)
    specs.append({
        "name": "matmul_2048",
        "make_inputs": lambda device: (
            torch.randn(2048, 2048, device=device, dtype=torch.float32),
            torch.randn(2048, 2048, device=device, dtype=torch.float32),
        ),
        "op": lambda a, b: torch.matmul(a, b),
    })

    # 2. Conv1d (Específicamente para tu caso de GALAH)
    specs.append({
        "name": "conv1d_spectroscopy_style",
        "make_inputs": lambda device: (
            torch.randn(64, 1, 16384, device=device, dtype=torch.float32),
            torch.randn(32, 1, 5, device=device, dtype=torch.float32),
            torch.randn(32, device=device, dtype=torch.float32),
        ),
        "op": lambda x, w, b: torch.nn.functional.conv1d(x, w, b, stride=1, padding=2),
    })

    # 3. Elementwise large
    specs.append({
        "name": "add_1d_large",
        "make_inputs": lambda device: (
            torch.randn(4_000_000, device=device, dtype=torch.float32),
            torch.randn(4_000_000, device=device, dtype=torch.float32),
        ),
        "op": lambda a, b: a + b,
    })

    return specs

# -----------------------------
# Performance Matrix: CPU vs XPU
# -----------------------------
def run_performance_matrix(n_warmup=15, n_iters=100, atol=1e-4):
    specs = make_operation_specs()

    has_xpu = torch.xpu.is_available()
    devices = ["cpu"] + (["xpu"] if has_xpu else [])

    all_rows = []

    for spec in specs:
        per_device = {}
        for device in devices:
            result = benchmark_op(
                op_fn=spec["op"],
                make_inputs_fn=spec["make_inputs"],
                device_str=device,
                n_warmup=n_warmup,
                n_iters=n_iters,
            )
            per_device[device] = result

        row = {
            "operation": spec["name"],
            "cpu_ms": per_device["cpu"]["mean_ms"],
        }

        if has_xpu:
            row["xpu_ms"] = per_device["xpu"]["mean_ms"]
            row["speedup_xpu"] = per_device["cpu"]["mean_ms"] / per_device["xpu"]["mean_ms"]
            
            # Validación de precisión entre dispositivos
            cpu_out = per_device["cpu"]["output"]
            xpu_out = per_device["xpu"]["output"]
            if torch.is_tensor(cpu_out) and torch.is_tensor(xpu_out):
                row["max_abs_diff"] = (cpu_out - xpu_out).abs().max().item()

        all_rows.append(row)

    return pd.DataFrame(all_rows)

# -----------------------------
# Ejecución
# -----------------------------
if __name__ == "__main__":
    print("Iniciando Benchmark en Intel Lunar Lake...")
    df_bench = run_performance_matrix()
    print(df_bench.to_string(index=False))

Iniciando Benchmark en Intel Lunar Lake...
                operation     cpu_ms   xpu_ms  speedup_xpu  max_abs_diff
              matmul_2048 104.950914 5.304210    19.786342    314.455383
conv1d_spectroscopy_style  46.228348 5.850957     7.900990     24.004822
             add_1d_large   1.620002 0.593605     2.729091     10.057915
